In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import pandas as pd
import pickle
import copy

from imports import *
from config import dir_config, ephys_config
from src.utils import dpca_utils

In [ ]:
compiled_dir  = Path(dir_config.data.compiled)
processed_dir = Path(dir_config.data.processed)

## Load data

Data loading stays in the notebook. The utils start after you have
`session_metadata`, `neuron_metadata`, `ephys`, and the trial-info dicts.

In [ ]:
session_to_exclude = ["210210_GP_JP", "241209_GP_TZ"]

session_metadata = pd.read_csv(Path(processed_dir, "sessions_metadata.csv"))
session_metadata = session_metadata[~session_metadata.session_id.isin(session_to_exclude)].reset_index(drop=True)

neuron_metadata = pd.read_csv(Path(processed_dir, "neuron_metadata.csv"))
neuron_metadata = neuron_metadata[~neuron_metadata.session_id.isin(session_to_exclude)].reset_index(drop=True)

with open(Path(processed_dir, "glm_hmm_models", "glm_hmm_masked_final.pkl"), "rb") as f:
    glm_hmm = pickle.load(f)
glm_hmm_original = copy.deepcopy(glm_hmm)

with open(Path(processed_dir, "ephys_neuron_wise.pkl"), "rb") as f:
    ephys = pickle.load(f)

## Extract trial info (blocks or glm-hmm states)

In [ ]:
data = glm_hmm["data"]

# HMM states — also flips sign for awayRF sessions in-place on glm_hmm["data"]
# compiled_dir loads reaction_time from trial CSVs (required by get_trial_num)
biased_state_trial_info, unbiased_state_trial_info, state_occupancy = \
    dpca_utils.extract_hmm_state_trial_info(session_metadata, glm_hmm_original, data,
                                            compiled_dir=compiled_dir)

# Blocks from prob_toRF (run after extract_hmm_state_trial_info so awayRF sign flip is applied)
equal_block_trial_info, unequal_block_trial_info = \
    dpca_utils.extract_block_trial_info(data, session_metadata["session_id"])

## Shared setup

In [ ]:
toRF_sessions  = session_metadata.session_id[session_metadata.prior_direction == "toRF"]
awayRF_sessions = session_metadata.session_id[session_metadata.prior_direction == "awayRF"]

alignments = list(ephys_config["alignment_settings_GP"].keys())  # ['baseline','visual','cue','response']
marginalization_keys = ['b', 's', 'c', 't']

condition_dict_states = {
    "state_values": ["biased", "unbiased"],
    "coherences":   [0, 0.06, 0.2, 0.5],
    "choices":      ["awayRF", "toRF"],
}

state_trial_info = {
    "biased":   biased_state_trial_info,
    "unbiased": unbiased_state_trial_info,
}

### Plot utils

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

def plot_variance_explained(dpca_results, alignments, margs_to_plot, n_components=3):
    fig, axes = plt.subplots(len(alignments), len(margs_to_plot),
                            figsize=(4 * len(margs_to_plot), 3 * len(alignments)),
                            sharey=True)

    for row, alignment in enumerate(alignments):
        evr = dpca_results[alignment]["model"].explained_variance_ratio_
        for col, marg in enumerate(margs_to_plot):
            ax = axes[row, col]
            cumvar = np.cumsum(evr[marg]) / np.sum(evr[marg])
            ax.plot(np.arange(1, n_components + 1), cumvar, marker="o")
            ax.axhline(0.95, color="r", linestyle="--", linewidth=1)
            ax.set_ylim(0, 1.1)
            ax.set_xticks(np.arange(1, n_components + 1))
            if row == 0:
                ax.set_title(marg_labels[marg])
            if col == 0:
                ax.set_ylabel(align_labels[alignment])
            ax.spines["top"].set_visible(False)
            ax.spines["right"].set_visible(False)

    fig.suptitle("Cumulative explained variance by marginalization (red = 95%)", y=1.01)
    plt.tight_layout()
    plt.show()

# ── Condition colors ───────────────────────────────────────────────────────────
# rows = coherence index (0–3), cols = state (0=biased, 1=unbiased)
biased_colors   = ["#96D6EC", "#6FC3EB", "#5289C6", "#4469B1"]
unbiased_colors = ["#F2A448", "#EF8D41", "#EC6A50", "#AC3626"]

plot_params = {
    "color":     {0: {0: biased_colors[0],   1: unbiased_colors[0]},
                  1: {0: biased_colors[1],   1: unbiased_colors[1]},
                  2: {0: biased_colors[2],   1: unbiased_colors[2]},
                  3: {0: biased_colors[3],   1: unbiased_colors[3]}},
    "linestyle": {0: "-", 1: "--"},   # choice: 0=awayRF solid, 1=toRF dashed
}

marg_labels = {"b": "Bias", "s": "Stimulus", "c": "Choice", "t": "Time"}
align_labels = {"baseline": "Baseline", "visual": "Visual", "cue": "Cue", "response": "Response"}


def plot_component(ax, time, data, axes_to_loop=None, plot_mean=False,
                   color=None, linestyle=None, alpha=1, lw=2, ylim=None):
    """Plot condition traces for one PC of one marginalization.

    Parameters
    ----------
    ax            : matplotlib Axes
    time          : 1-D array of ms time points
    data          : ndarray (n_states, n_coh, n_choices, n_time)  — Z[marg][PC, :]
    axes_to_loop  : subset of ['b','s','c'] to draw as separate lines;
                    non-looped axes are averaged over
    plot_mean     : if True, average over non-looped condition axes
    """
    all_cond_axes = ['b', 's', 'c']
    if axes_to_loop is None:
        axes_to_loop = all_cond_axes
    axis_pos = {a: i for i, a in enumerate(all_cond_axes)}
    avg_axes = tuple(axis_pos[a] for a in all_cond_axes if a not in axes_to_loop)

    for b in (range(2) if 'b' in axes_to_loop else [0]):
        for s in ([0,3] if 's' in axes_to_loop else [0]):
            for c in (range(2) if 'c' in axes_to_loop else [0]):
                if plot_mean:
                    idx = tuple(
                        b if a == 'b' else s if a == 's' else c if a == 'c' else slice(None)
                        for a in all_cond_axes
                    ) + (slice(None),)
                    y = np.nanmean(data[idx], axis=tuple(range(len(avg_axes))))
                else:
                    y = data[b, s, c, :]

                ax.plot(
                    time, y.reshape(-1),
                    color=plot_params["color"][s][b] if color is None else color,
                    linestyle=plot_params["linestyle"][c] if linestyle is None else linestyle,
                    linewidth=lw,
                    alpha=alpha,
                )

    ax.axvline(0, color="black", linestyle="--", linewidth=1)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    if ylim:
        ax.set_ylim(ylim)

        
def plot_self_projection(projections, time_axes, alignments, margs_to_plot, axes_per_marg=None):
    fig, axes = plt.subplots(len(margs_to_plot), len(alignments),
                            figsize=(5 * len(alignments), 4 * len(margs_to_plot)),
                            )#sharey="row")

    for col, alignment in enumerate(alignments):
        Z    = projections[alignment][alignment]   # self-projection
        time = time_axes[alignment]
        for row, marg in enumerate(margs_to_plot):
            ax = axes[row, col]
            # plot_component(ax, time, Z[marg][PC], axes_to_loop=axes_per_marg[marg], plot_mean=False, lw=2)
            plot_component(ax, time, Z[marg][PC], axes_to_loop=None, plot_mean=False, lw=2)
            if row == 0:
                ax.set_title(align_labels[alignment], fontsize=13)
            if col == 0:
                ax.set_ylabel(f"{marg_labels[marg]} PC{PC+1}", fontsize=12)

    # legend
    handles = (
        [plt.Line2D([], [], color=biased_colors[i],   label=f"{coh*100:.0f}% biased")   for i, coh in enumerate([0, 0.06, 0.2, 0.5])] +
        [plt.Line2D([], [], color=unbiased_colors[i], label=f"{coh*100:.0f}% unbiased") for i, coh in enumerate([0, 0.06, 0.2, 0.5])] +
        [plt.Line2D([], [], color="k", linestyle="-",  label="awayRF choice"),
        plt.Line2D([], [], color="k", linestyle="--", label="toRF choice")]
    )
    fig.legend(handles=handles, bbox_to_anchor=(1.01, 0.9), loc="upper left", fontsize=10)
    fig.suptitle("Self-projection: PC1 by marginalization and alignment", y=1.01)
    plt.tight_layout()
    plt.show()

def plot_cross_projection(projections, time_axes, alignments, margs_to_plot, axes_per_marg=None):
    fig, axes = plt.subplots(len(margs_to_plot), len(alignments),
                            figsize=(5 * len(alignments), 4 * len(margs_to_plot)),
                            )#sharey="row")

    for row, marg in enumerate(margs_to_plot):
        for col, proj_align in enumerate(alignments):
            ax   = axes[row, col]
            Z    = projections[fit_align][proj_align]
            time = time_axes[proj_align]

            plot_component(ax, time, Z[marg][PC], axes_to_loop=["b", "c"], plot_mean=False, lw=2)

            if row == 0:
                ax.set_title(f"Project: {align_labels[proj_align]}", fontsize=12)
            if col == 0:
                ax.set_ylabel(f"{marg_labels[marg]} PC{PC+1}", fontsize=12)

            # highlight self-projection column (baseline fit → baseline project)
            if proj_align == fit_align:
                for spine in ax.spines.values():
                    spine.set_edgecolor("#e05c00")
                    spine.set_linewidth(2)

    handles = (
        [plt.Line2D([], [], color=biased_colors[i],   label=f"{coh*100:.0f}% biased")   for i, coh in enumerate([0, 0.06, 0.2, 0.5])] +
        [plt.Line2D([], [], color=unbiased_colors[i], label=f"{coh*100:.0f}% unbiased") for i, coh in enumerate([0, 0.06, 0.2, 0.5])] +
        [plt.Line2D([], [], color="k", linestyle="-",  label="awayRF choice"),
        plt.Line2D([], [], color="k", linestyle="--", label="toRF choice")]
    )
    fig.legend(handles=handles, bbox_to_anchor=(1.01, 0.9), loc="upper left", fontsize=10)
    fig.suptitle(f"Cross-period projection — fit on {align_labels[fit_align]}  (orange border = self-projection)",
                fontsize=13, y=1.01)
    plt.tight_layout()
    plt.show()

---
## Example 1 — All neurons, HMM states (baseline)

In [ ]:
neuron_ids = dpca_utils.get_neuron_ids(neuron_metadata, toRF_sessions)

avg, tw = dpca_utils.create_dpca_matrix(
    toRF_sessions, condition_dict_states, neuron_ids,
    state_trial_info, neuron_metadata, ephys, ephys_config,
    condition_type="states",
)
fit_avg, fit_tw, full_avg, full_tw = dpca_utils.clean_dpca_data(avg, tw, alignments)

dpca_results  = dpca_utils.fit_dpca_all_alignments(fit_avg, fit_tw, alignments, marginalization_keys=marginalization_keys)
projections   = dpca_utils.cross_period_projection(dpca_results, fit_avg, alignments)
time_axes     = dpca_utils.build_time_axes(fit_avg, ephys_config)

# projections[fit_alignment][project_alignment][marginalization_key]
# e.g. baseline-fit axes applied to response-period data:
Z_baseline_on_response = projections["baseline"]["response"]

Plot variance explained, self-projection and cross-projection

In [ ]:
margs_to_plot = ['b', 's', 'c', 't']
n_components  = 3
plot_variance_explained(dpca_results, alignments, margs_to_plot, n_components=n_components)


In [ ]:

PC = 0
margs_to_plot = ['b', 's', 'c']
axes_per_marg = {'b': ['b', 'c'], 's': ['s', 'c'], 'c': ['c']}  # which condition axes vary per plot
plot_self_projection(projections, time_axes, alignments, margs_to_plot)#, axes_per_marg)
PC=1
plot_self_projection(projections, time_axes, alignments, margs_to_plot)#, axes_per_marg)


In [ ]:
fit_align = "baseline"
margs_to_plot = ['b', 's', 'c']
plot_cross_projection(projections, time_axes, alignments, margs_to_plot, axes_per_marg)


---
## Example 2 — Exclude trash and undefined (trash excluded by default; add undefined)

In [ ]:
neuron_ids_no_trash = dpca_utils.get_neuron_ids(
    neuron_metadata, toRF_sessions,
    exclude_cell_types=["trash", "undefined"],
)

avg, tw = dpca_utils.create_dpca_matrix(
    toRF_sessions, condition_dict_states, neuron_ids_no_trash,
    state_trial_info, neuron_metadata, ephys, ephys_config,
    condition_type="states",
)
fit_avg, fit_tw, full_avg, full_tw = dpca_utils.clean_dpca_data(avg, tw, alignments)

dpca_results_no_undefined = dpca_utils.fit_dpca_all_alignments(fit_avg, fit_tw, alignments)
projections_no_undefined  = dpca_utils.cross_period_projection(dpca_results_no_undefined, fit_avg, alignments)

#### plot results

In [ ]:
margs_to_plot = ['b', 's', 'c', 't']
plot_variance_explained(dpca_results_no_undefined, alignments, margs_to_plot, n_components=n_components)
PC = 0
margs_to_plot = ['b', 's', 'c']
axes_per_marg = {'b': ['b', 'c'], 's': ['s', 'c'], 'c': ['c']}
plot_self_projection(projections_no_undefined, time_axes, alignments, margs_to_plot, axes_per_marg)
fit_align = "baseline"
margs_to_plot = ['b', 's', 'c']
plot_cross_projection(projections_no_undefined, time_axes, alignments, margs_to_plot, axes_per_marg)


---
## Example 3 — 10% neuron leave-out (repeated for stability)

Each repeat randomly drops 10% of neurons. Pass the same `rng` seed for
reproducibility, or a different seed per repeat for a stability analysis.

In [ ]:
N_REPEATS = 10
all_projections_leaveout = []

for repeat in range(N_REPEATS):
    neuron_ids_leaveout = dpca_utils.get_neuron_ids(
        neuron_metadata, toRF_sessions,
        leave_out_fraction=0.1,
        rng=np.random.default_rng(repeat),
    )

    avg, tw = dpca_utils.create_dpca_matrix(
        toRF_sessions, condition_dict_states, neuron_ids_leaveout,
        state_trial_info, neuron_metadata, ephys, ephys_config,
        condition_type="states",
    )
    fit_avg, fit_tw, full_avg, full_tw = dpca_utils.clean_dpca_data(avg, tw, alignments)

    results   = dpca_utils.fit_dpca_all_alignments(fit_avg, fit_tw, alignments)
    projections = dpca_utils.cross_period_projection(results, fit_avg, alignments)
    all_projections_leaveout.append(projections)

---
## Example 4 — Fit on half the trials, project onto the held-out half

Useful for the saccade-onset (response) period or any alignment.
`split_trial_info_half` splits each session independently within each state/block.

In [ ]:
half1_trial_info, half2_trial_info = dpca_utils.split_trial_info_half(
    state_trial_info, toRF_sessions, seed=0
)

neuron_ids = dpca_utils.get_neuron_ids(neuron_metadata, toRF_sessions)

# build matrices for each half
avg_fit,  tw_fit  = dpca_utils.create_dpca_matrix(
    toRF_sessions, condition_dict_states, neuron_ids,
    half1_trial_info, neuron_metadata, ephys, ephys_config,
    condition_type="states",
)
avg_test, tw_test = dpca_utils.create_dpca_matrix(
    toRF_sessions, condition_dict_states, neuron_ids,
    half2_trial_info, neuron_metadata, ephys, ephys_config,
    condition_type="states",
)

# clean fit data (strict: drop any-NaN timepoints)
fit_avg, fit_tw, _, _ = dpca_utils.clean_dpca_data(avg_fit, tw_fit, alignments)
# clean test data (lenient: drop only all-NaN timepoints)
_, _, full_avg_test, _ = dpca_utils.clean_dpca_data(avg_test, tw_test, alignments)

# fit dPCA on saccade-onset period using half-1 trials
dpca_response = dpca_utils.fit_dpca_on_alignment(fit_avg["response"], fit_tw["response"])
response_model = dpca_response[0]  # fitted dPCA model

# project held-out half-2 trials onto the fitted axes, across all periods
_, Z_baseline_heldout  = dpca_utils.dpca_transform(response_model, full_avg_test["baseline"])
_, Z_visual_heldout    = dpca_utils.dpca_transform(response_model, full_avg_test["visual"])
_, Z_cue_heldout       = dpca_utils.dpca_transform(response_model, full_avg_test["cue"])
_, Z_response_heldout  = dpca_utils.dpca_transform(response_model, full_avg_test["response"])

---
## Example 5 — Blocks instead of HMM states

Block membership comes from `glm_hmm["data"][session_id]["prob_toRF"]`:
- `prob_toRF == 50` → equal block
- `prob_toRF != 50` → unequal block

In [ ]:
condition_dict_blocks = {
    "state_values": ["equal", "unequal"],
    "coherences":   [0, 0.06, 0.2, 0.5],
    "choices":      ["awayRF", "toRF"],
}

block_trial_info = {
    "equal":   equal_block_trial_info,
    "unequal": unequal_block_trial_info,
}

neuron_ids = dpca_utils.get_neuron_ids(neuron_metadata, toRF_sessions)

avg, tw = dpca_utils.create_dpca_matrix(
    toRF_sessions, condition_dict_blocks, neuron_ids,
    block_trial_info, neuron_metadata, ephys, ephys_config,
    condition_type="blocks",
)
fit_avg, fit_tw, full_avg, full_tw = dpca_utils.clean_dpca_data(avg, tw, alignments)

dpca_results_blocks = dpca_utils.fit_dpca_all_alignments(fit_avg, fit_tw, alignments)
projections_blocks  = dpca_utils.cross_period_projection(dpca_results_blocks, full_avg, alignments)

---
## Example 6 — Combining filters: exclude trash, 10% leave-out, blocks

In [ ]:
neuron_ids_combined = dpca_utils.get_neuron_ids(
    neuron_metadata, toRF_sessions,
    exclude_cell_types=["trash"],
    leave_out_fraction=0.1,
    rng=np.random.default_rng(0),
)

avg, tw = dpca_utils.create_dpca_matrix(
    toRF_sessions, condition_dict_blocks, neuron_ids_combined,
    block_trial_info, neuron_metadata, ephys, ephys_config,
    condition_type="blocks",
)
fit_avg, fit_tw, full_avg, full_tw = dpca_utils.clean_dpca_data(avg, tw, alignments)

dpca_results_combined = dpca_utils.fit_dpca_all_alignments(fit_avg, fit_tw, alignments)
projections_combined  = dpca_utils.cross_period_projection(dpca_results_combined, full_avg, alignments)